# DeBERTa-v3-base: Medication Sentiment

Train a classifier to predict sentiment towards the medication named in each Reddit post.

Run the sections in order. The notebook uses the existing `training_data.csv`, `validation_data.csv`, and `test_data.csv` files. Model selection uses validation macro-F1; the test set is reserved for the final evaluation.

Each row must contain one target medication and its sentiment label. Predictions represent Reddit sentiment, not clinical effectiveness.

## 1. Configuration

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

DATA_DIR = Path(".")
MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = Path("deberta_model")

SEED = 123
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
EPOCHS = 3
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

set_seed(123)


## 2. Load and prepare the data

In [3]:
def prepare_data(path):
    frame = pd.read_csv(path, dtype={"id": str})
    required = ["id", "drug", "title", "body", "label"]

    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

    frame = frame[required].copy()

    if frame[["id", "drug", "label"]].isna().any().any():
        raise ValueError(f"{path} contains missing IDs, medications, or labels.")

    for column in ["id", "drug", "label"]:
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"{path} contains empty values in {column}.")

    frame["text"] = (
        frame["title"].fillna("").astype(str)
        + "\n"
        + frame["body"].fillna("").astype(str)
    )
    if frame["text"].str.strip().eq("").any():
        raise ValueError(f"{path} contains empty post text.")

    return frame


train = prepare_data(DATA_DIR / "training_data.csv")
validation = prepare_data(DATA_DIR / "validation_data.csv")
test = prepare_data(DATA_DIR / "test_data.csv")

for name_a, data_a, name_b, data_b in [
    ("training", train, "validation", validation),
    ("training", train, "test", test),
    ("validation", validation, "test", test),
]:
    if set(data_a["id"]) & set(data_b["id"]):
        raise ValueError(f"Post IDs overlap between {name_a} and {name_b}.")

label_names = sorted(train["label"].unique())
if len(label_names) < 2:
    raise ValueError("Training data must contain at least two sentiment classes.")

label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

for frame in (train, validation, test):
    unknown = set(frame["label"]) - set(label_names)
    if unknown:
        raise ValueError(f"Labels not present in training data: {unknown}")
    frame["labels"] = frame["label"].map(label2id)

## 3. Tokenise the posts

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def tokenise(batch):
    medication = ["Target medication: " + drug for drug in batch["drug"]]
    return tokenizer(
        medication,
        batch["text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )


def make_dataset(frame):
    dataset = Dataset.from_pandas(
        frame[["drug", "text", "labels"]],
        preserve_index=False,
    )
    return dataset.map(
        tokenise,
        batched=True,
        remove_columns=["drug", "text"],
    )


train_dataset = make_dataset(train)
validation_dataset = make_dataset(validation)

Map:   0%|          | 0/7452 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

## 4. Train the model

In [5]:
def compute_metrics(prediction):
    predicted = np.argmax(prediction.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "macro_f1": f1_score(
            prediction.label_ids,
            predicted,
            labels=list(id2label),
            average="macro",
            zero_division=0,
        ),
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    label2id=label2id,
    id2label=id2label,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
validation_metrics = trainer.evaluate()
validation_metrics

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.893800,0.760685,0.676939,0.643364
2,0.645800,0.792583,0.758767,0.735251
3,0.535100,0.980140,0.759830,0.738215


{'eval_loss': 0.9801402688026428,
 'eval_accuracy': 0.7598299681190224,
 'eval_macro_f1': 0.7382147365883784,
 'eval_runtime': 16.2894,
 'eval_samples_per_second': 57.768,
 'eval_steps_per_second': 28.915,
 'epoch': 3.0}

## 5. Save the model

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

('deberta_model\\tokenizer_config.json',
 'deberta_model\\special_tokens_map.json',
 'deberta_model\\spm.model',
 'deberta_model\\added_tokens.json')

## 6. Final test evaluation

Run this section once, after selecting the model using validation macro-F1.

In [9]:
test_dataset = make_dataset(test)
prediction = trainer.predict(test_dataset)

predicted_ids = np.argmax(prediction.predictions, axis=-1)
print(classification_report(
    test["labels"],
    predicted_ids,
    labels=list(id2label),
    target_names=label_names,
    zero_division=0,
))

print("Test metrics:", prediction.metrics)

Map:   0%|          | 0/927 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

    negative       0.79      0.84      0.81       444
     neutral       0.73      0.70      0.71       312
    positive       0.62      0.57      0.60       171

    accuracy                           0.74       927
   macro avg       0.71      0.70      0.71       927
weighted avg       0.74      0.74      0.74       927

Test metrics: {'test_loss': 1.0101070404052734, 'test_accuracy': 0.7411003236245954, 'test_macro_f1': 0.7070878198938765, 'test_runtime': 17.3025, 'test_samples_per_second': 53.576, 'test_steps_per_second': 26.817}
